# 14e · 位点地图：每个位置是干什么的

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F]
                 ├─ 14a 稳定性换成 kcal/mol ✅
                 ├─ 14b 从测序读数到适应度 ✅
                 ├─ 14c 一个数拆成两个能量 ✅
                 ├─ 14d 少量数据 + 主动学习 ✅
                 ├─ 14e 位点地图                 ← 你在这里
                 ├─ 14f 药理伴侣
                 └─ 14g 门面和讲法
```

## 背景：从"这个突变怎么样"到"这个位置是干什么的"

前面几章都是在回答**突变**的问题。但药物研发更常问的是**位置**的问题：

> 这个靶点上，哪些地方是结构支架（动不得）、哪些地方是干活的（可以当药物口袋）、哪些地方随便改（可以挂标签、做改造）？

14c 的两个能量可以直接回答这个。把一个位置上 19 种替换的能量取平均，得到这个**位置**的两个代价：

| 位置类型 | 折叠代价 | 功能代价 | 这个位置是什么 | 对项目意味着什么 |
|---|---|---|---|---|
| **结构关键** | 高 | 低 | 蛋白的骨架，埋在里面 | 动不得；突变坏了可以试试伴侣分子扶住它 |
| **功能关键** | 低 | 高 | 折得好好的，但这里一动就不工作 | **活性位点、配体口袋、结合界面——候选药物口袋** |
| **两者都关键** | 高 | 高 | 又埋又重要 | 典型的催化中心 |
| **可以改** | 低 | 低 | 表面、不重要 | 挂荧光标签、改溶解度、做改造的地方 |

"**功能关键**"这一类是重点：它是"蛋白好好的，但不工作"的位置，也是小分子有可能结合的地方。

## 这一章做什么

| 步骤 | 内容 |
|---|---|
| 1 | 给一批蛋白的每个位置算两个代价，贴标签 |
| 2 | **对不对**：和 UniProt 注释的活性位点、配体位点比 |
| 3 | **有没有用**：三个**已经成药**的变构口袋，模型事先点不点得亮 |
| 4 | 地图进数据湖，能用 SQL 查 |

---

## 准备

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from qafi_platform import athena, lake, tracking
from qafi_platform.lakehouse import sql
from qafi_core import energy, sites, stability, validation
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
WORK, MODELS, MAP = ROOT / "data/stability", ROOT / "data/models", ROOT / "data/sites"
MAP.mkdir(parents=True, exist_ok=True)
print("处理器:", stability.device())

---

## 第 1 步 · 给每个位置贴标签

### 在干什么
用 14c 的模型，给一批蛋白的**每个位置的每一种替换**算两个能量，再按位置平均，贴上标签。

### 哪些蛋白
- **108 个锁死的考试蛋白**：模型没见过，标签才算数
- 加上三个**有成药变构口袋**的蛋白：KRAS、ABL1、SHP2（第 3 步要用）

### 怎么定"高"和"低"
阈值**按蛋白各算各的**（每个蛋白最差的 25%）。因为蛋白之间的整体容忍度差很多，而这里要问的是"在这个蛋白**内部**，哪些位置突出"。

### 执行
训练模型（和 14c 一样），算特征，给约 200 万个替换打分。约 10 分钟。

In [ ]:
POCKETS = {    # 已上市或在临床的药结合的口袋，残基取自沉积的结构
    "P01116": ("KRAS switch-II（sotorasib，6OIM）",
               [9, 10, 12, 16, 34, 58, 59, 60, 61, 62, 63, 68, 72, 95, 96, 99, 103]),
    "P00519": ("ABL1 肉豆蔻酸口袋（asciminib，5MO4）",   # 结构用 1b 编号 = UniProt 1a + 19
               [332, 337, 340, 341, 344, 429, 432, 433, 434, 435, 462, 464, 465, 468, 493, 502, 506, 510]),
    "Q06124": ("SHP2 通道（SHP099，5EHR）",
               [108, 110, 111, 113, 114, 218, 219, 249, 250, 253, 254, 257, 491, 492, 495]),
}
splits = stability.load_splits(WORK / "mega_splits.pkl")
mega = stability.add_features(
    stability.megascale(WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv", splits),
    pd.read_parquet(WORK / "structure.parquet"), pd.read_parquet(WORK / "residue_index.parquet"),
    np.load(WORK / "logprobs.npy"))
mega = mega[mega.split.isin(["train", "val"])]
split_table = pd.read_csv(ROOT / "dbt/seeds/protein_split.csv")
sequences = pd.read_parquet(MODELS / "proteome_sequences.parquet").set_index("uniprot_id").sequence
locked = [sequences[u] for u in split_table.loc[split_table.split == "test", "uniprot_id"] if u in sequences.index]
mega = mega[~mega.domain.isin({d for d, s in mega.groupby("domain").seq.first().items()
                               if any(validation.related(s, t) for t in locked)})]
labels = pd.read_parquet(MODELS / "train_variants.parquet")
titles = sql(w, "SELECT dataset_id, title FROM qafi_dev.gold.dim_label_dataset").set_index("dataset_id").title.to_dict()
labels["reads"] = [energy.readout_kind(k, titles.get(d))
                   for k, d in zip(labels.data_kind, labels.dataset_id, strict=True)]
kinds = labels.groupby("uniprot_id").reads.apply(set)
paired = {p for p, k in kinds.items() if {energy.ABUNDANCE, energy.ACTIVITY} <= k}
rows, vectors = energy.stack(mega, np.load(WORK / "vectors.npy"), labels,
                             pd.read_parquet(MODELS / "train_embeddings.parquet"),
                             np.asarray(np.load(MODELS / "train_embeddings.npy", mmap_mode="r")), titles)
readout = rows[(rows.kind == energy.DDG) | rows.protein.isin(paired)]
models = [energy.fit(readout, vectors, energy.FEATURES, seed=s) for s in (0, 1, 2)]
print(f"模型训练完成（{len(paired)} 个成对蛋白的读数层）")

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "databricks" / "src"))
from lakehouse import language_model as lm
wanted = sorted(set(POCKETS) | set(split_table.loc[split_table.split == "test", "uniprot_id"]))
residues = pd.read_parquet(MODELS / "proteome_residues.parquet", filters=[("uniprot_id", "in", wanted)])
have = sorted(residues.uniprot_id.unique())
seqs = [sequences[u] for u in have]
model, tokenizer = lm.load()
read = lm.read_proteins(seqs, model, tokenizer, embeddings=True)
logprobs = np.concatenate([r[0] for r in read])
protein_vectors = np.concatenate([r[1] for r in read]).astype(np.float16)
index = pd.DataFrame({"domain": np.repeat(have, [len(s) for s in seqs]),
                      "position": np.concatenate([np.arange(1, len(s) + 1) for s in seqs])})
structure = residues[["uniprot_id", "position", "aa", "plddt", "rsa", "contacts_8", "contacts_12",
                      "hse_up", "hse_down"]].rename(columns={"uniprot_id": "domain"})
every = pd.DataFrame({
    "domain": np.repeat(index.domain.values, 20), "position": np.repeat(index.position.values, 20),
    "wt_aa": np.repeat(index.merge(structure, on=["domain", "position"], how="left").aa.values, 20),
    "mut_aa": np.tile(list(energy.AMINO_ACIDS), len(index))}).dropna(subset=["wt_aa"])
every = every[every.wt_aa != every.mut_aa].assign(ddg=0.0)
every = stability.add_features(every, structure, index, logprobs, strict=False)
fold, func = [], []
for start in range(0, len(every), 200_000):
    part = every.iloc[start:start + 200_000]
    f, g = zip(*(energy.energies(m, s, part, protein_vectors, energy.FEATURES) for m, s, _ in models))
    fold.append(np.mean(f, axis=0)); func.append(np.mean(g, axis=0))
every["ddg_fold"], every["ddg_func"] = np.concatenate(fold), np.concatenate(func)
every = every.rename(columns={"domain": "uniprot_id"})
positions = sites.classify(every)
positions.to_parquet(MAP / "positions.parquet")
print(f"{len(have)} 个蛋白、{len(positions):,} 个位置、{len(every):,} 个替换")
positions.position_is.value_counts().rename("位置数")

### 结果怎么看
118 个蛋白、103,744 个位置、197 万个替换。四类的数量：可以改最多（约 7 万），结构关键和功能关键各约 8 千，两者都关键约 1.8 万。

按定义每类的比例是固定的（各取最差 25%），所以数量本身不说明问题——**下面两步才是检验**。

---

## 第 2 步 · 对不对：和 UniProt 的注释比

### 在干什么
UniProt 为人工审核过的蛋白标注了**活性位点**、**配体结合位点**和其他功能位点。我们的"功能关键"位置，应该比别的位置更容易落在这些注释上。

### 这是什么性质的检验
这是**事后验证**，不是训练目标——模型从没见过任何位点注释，它只见过突变的测量值。如果"功能关键"确实富集在注释位点上，说明两个能量的拆分抓到了真东西。

### 执行

In [ ]:
uniprot = pd.read_csv("https://rest.uniprot.org/uniprotkb/stream?query=reviewed:true+AND+organism_id:9606"
                      "&fields=accession,ft_act_site,ft_binding,ft_site&format=tsv", sep="\t")
annotated = sites.known_sites(uniprot)
print(f"UniProt 注释了 {len(annotated):,} 个功能位置，覆盖 {annotated.uniprot_id.nunique():,} 个蛋白")
sites.enrichment(positions, annotated).round(4)

### 结果怎么看

| 位置类型 | 位置数 | 其中有注释 | 比例 | 比值比 | p |
|---|---|---|---|---|---|
| **两者都关键** | 17,831 | 186 | **1.04%** | **9.33** | ≈0 |
| **功能关键** | 8,153 | 53 | **0.65%** | **2.71** | ≈0 |
| 结构关键 | 8,153 | 5 | 0.06% | 0.21 | 1.0 |
| 可以改 | 69,607 | 39 | 0.06% | 0.08 | 1.0 |

方向完全正确：

- **"两者都关键"富集最强**（9.3 倍）。这符合生物学——催化中心通常既埋得深又不能动，两个能量都高。
- **"功能关键"富集 2.7 倍**，而**"结构关键"反而被稀释**（0.21 倍）。这正是拆分有意义的证据：如果两个能量只是同一个数的两个副本，它们对注释位点的富集应该一样，而不是一个富集一个稀释。
- "可以改"几乎不含注释位点（0.08 倍）——它确实是可以动的地方。

绝对比例都很低（1% 量级），因为 UniProt 只给一小部分位置做了注释（10 万个位置里只有 283 个有注释）。**比值比才是要看的数字。**

### 面试怎么说
> "The two energies were never shown a site annotation, yet positions the model calls functionally costly are enriched 2.7-fold for UniProt active and binding sites, those costly in both energies 9.3-fold, while structurally costly positions are depleted. A single score cannot produce that split."

---

## 第 3 步 · 有没有用：已经成药的变构口袋

### 在干什么
最硬的检验：拿三个**已经有药物结合**的变构口袋，看模型在不知情的情况下，会不会把这些残基排在"功能敏感"的前面。

### 三个口袋

| 蛋白 | 口袋 | 药 | 为什么是好例子 |
|---|---|---|---|
| **KRAS** | switch-II 口袋 | sotorasib（已上市） | 几十年"不可成药"的靶点，靠这个口袋破局 |
| **ABL1** | 肉豆蔻酸口袋 | asciminib（已上市） | 变构抑制剂，不碰 ATP 口袋，所以不怕常见耐药突变 |
| **SHP2** | 三结构域交界的通道 | SHP099（临床） | 变构位点的典型 |

残基编号取自沉积的晶体结构（配体 4 Å 以内的重原子接触）。**注意 ABL1**：结构用的是 1b 异构体编号，比 UniProt 的 1a 大 19，这里已经换算过——这类编号错位是最容易悄悄毁掉一个验证的坑。

### 怎么算过关
把一个蛋白自己的所有位置按"功能代价"排序，看口袋残基排在什么百分位（0 = 最敏感，1 = 最不敏感）。口袋应该明显靠前。

### 执行

In [ ]:
found = []
for uniprot_id, (what, pocket) in POCKETS.items():
    out = sites.pocket_rank(positions, uniprot_id, pocket)
    found.append({"口袋": what, "残基数": out["pocket residues"], "蛋白位置数": out["protein positions"],
                  "口袋的中位百分位": out["median percentile"],
                  "其他位置": out["median percentile elsewhere"], "p": out["p"]})
pd.DataFrame(found).round(4)

### 结果怎么看

| 口袋 | 残基 | 口袋的中位百分位 | 其他位置 | p |
|---|---|---|---|---|
| **KRAS switch-II**（sotorasib） | 17 | **0.233** | 0.532 | **0.0012** |
| **ABL1 肉豆蔻酸口袋**（asciminib） | 18 | **0.235** | 0.506 | **0.0017** |
| SHP2 通道（SHP099） | 15 | 0.506 | 0.498 | 0.41 |

**三个里中了两个。** KRAS 和 ABL1 这两个口袋的残基，被模型排在自己蛋白里功能最敏感的前 23%——而模型只见过突变测量，从没见过任何药物、任何口袋。

SHP2 没有信号。这值得说清楚：SHP2 的通道位于**三个结构域的交界处**，它的作用是维持蛋白的自抑制构象，而不是直接参与催化或结合配体。我们的"功能能"学的是"折好之后还能不能干活"，对这种**构象开关**类型的位点天然不敏感。这是这套方法的一个真实边界。

### 面试怎么说
> "As a prospective check I took three allosteric pockets that marketed or clinical drugs bind and asked where their residues rank inside their own protein. KRAS switch-II and the ABL1 myristoyl pocket land in the top quarter by functional cost, p around 0.001, with the model having seen no drug and no pocket. SHP2's tunnel does not — it is a conformational switch at a domain junction, which is outside what a folded-and-functional model represents."

---

## 第 4 步 · 地图进数据湖

### 在干什么
把位置地图写进 S3 的 lake 桶，在 Glue 里登记成一张表，这样它和 10a 的预测表一样可以用 SQL 查。

### 背景
一张放在笔记本电脑上的 parquet 文件不是产品。放进湖里之后：分析师可以直接写 SQL（"TP53 上有哪些功能关键但结构可改的位置"）、API 可以查、agent 可以调用。这是把一章的科学结果**交付**出去。

表的定义写在 `infra/dev/lake.tf`（`sites`，和 `variant_effect`、`proteins` 并列），由 Terraform 创建。

### 执行
先建表（Terraform），再上传数据。

In [ ]:
run("terraform -chdir=infra/dev plan -no-color -out=sites.tfplan | tail -5")

In [ ]:
import boto3, json
run("terraform -chdir=infra/dev apply -no-color sites.tfplan | tail -3")
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
s3, client = aws.client("s3"), aws.client("athena")
buckets = json.loads(subprocess.run("terraform -chdir=infra/dev output -json bucket_names",
                                    shell=True, capture_output=True, text=True, check=True).stdout)
release = pd.Timestamp.today().strftime("%Y-%m-%d")
out = MAP / "sites.parquet"
positions.astype({"position": "int32", "fold_cost": "float32", "func_cost": "float32"}).to_parquet(out, index=False)
s3.upload_file(str(out), buckets["lake"], f"sites/release={release}/sites.parquet")
lake.query(client, f"ALTER TABLE sites ADD IF NOT EXISTS PARTITION (release = '{release}')",
           "qafi-dev", "qafi_dev_lake")
print(f"已上传 {len(positions):,} 行 → s3://{buckets['lake']}/sites/release={release}/")

In [ ]:
summary_sql = ("SELECT position_is, count(*) AS positions, round(avg(fold_cost), 3) AS mean_fold_cost, "
               "round(avg(func_cost), 3) AS mean_func_cost "
               "FROM sites WHERE release = ? GROUP BY position_is ORDER BY positions DESC")
# Athena reads an execution parameter as SQL, so a string has to arrive quoted: athena.literal does that
lake.query(client, summary_sql, "qafi-dev", "qafi_dev_lake", parameters=[athena.literal(release)])

### 结果怎么看
同一张地图，现在用 SQL 就能查。一个分析师想问"KRAS 上哪些位置功能关键但结构上可以改"，不用找我们要文件，直接：

```sql
SELECT position, func_cost FROM sites
WHERE release = '…' AND uniprot_id = 'P01116' AND position_is = 'functional'
ORDER BY func_cost
```

在 KRAS 上这条查询返回的前几位是 **75、17、12、38、117、35、123、13**。第 **12** 和第 **13** 位正是 KRAS 最著名的两个致癌突变热点，而第 12 位也在 sotorasib 的 switch-II 口袋残基名单里——模型没被告诉过任何一件事。

---

## 第 5 步 · 接进 API 和 agent

### 在干什么
让 10b 的对外 API 和 12 号的 agent 也能用这张地图。

### 背景
放进湖里，分析师能用 SQL 查。但 API 和 agent 面向的是不写 SQL 的人——以及其他程序。新增一个地址：

```
GET /v1/proteins/{protein}/sites?position_is=functional
```

和 10a 的做法一样，API 读的是**视图** `sites_current`，不是某个具体版本；换一版地图只要移动视图，不用动代码。agent 那边新增一个工具 `get_site_map`，并且给护栏的依据检查写了对应的文字（12 号的教训：JSON 判不准，要写成句子）。

实现：`src/qafi_platform/api.py` 加一个路由，`src/qafi_platform/agent.py` 加一个工具，各自配了测试（API 21 个、agent 14 个）。镜像重建后由 Terraform 按摘要部署。

### 执行

In [ ]:
import boto3, json
from qafi_platform import api_client
API = json.loads(subprocess.run("terraform -chdir=infra/dev output -json api",
                                shell=True, capture_output=True, text=True, check=True).stdout)
session = boto3.Session(profile_name="qafi", region_name="eu-central-1")
status, body, seconds = api_client.get(session, f"{API['url'].rstrip('/')}/v1/proteins/KRAS/sites")
print(f"{status} in {seconds:.1f}s | KRAS 共 {body['mapped']} 个位置")
pd.DataFrame(body["sites"]).head(8)

### 结果怎么看
线上 API 返回 KRAS 全部 189 个位置，按功能代价从高到低排好。

一个值得看的细节：KRAS 最著名的三个致癌突变热点 **G12、G13、Q61**，在这个排序里分别排第 **11、26、44** 位（前 6%、14%、23%）——第 12 位归为 mixed（两个能量都高），13 和 61 是 functional。模型只见过突变的测量值，从没被告知哪些位置和癌症有关。

agent 也能用了。问它"KRAS 哪些位置功能敏感但结构上能改"，它会调用 `get_site_map`，用返回的数字回答，并标注出处。

---

## 第 6 步 · 记账

In [ ]:
enrichment = sites.enrichment(positions, annotated).set_index("position_is")
pockets = pd.DataFrame(found)
metrics = {f"odds_ratio_{k}": float(enrichment.loc[k, "odds ratio"]) for k in enrichment.index
           if np.isfinite(enrichment.loc[k, "odds ratio"])}
metrics |= {f"pocket_percentile_{i}": float(r["口袋的中位百分位"]) for i, r in pockets.iterrows()}
metrics |= {"positions": len(positions), "proteins": int(positions.uniprot_id.nunique())}
tracking.experiment_run(
    EXPERIMENT, "site_map",
    params={"proteins": int(positions.uniprot_id.nunique()), "quantile": 0.25,
            "readout_proteins": len(paired), "pockets": len(POCKETS), "release": release},
    metrics=metrics,
    tables={"enrichment": enrichment.reset_index(), "pockets": pockets},
    chapter="14e")
print("已记录")

---

## 这一章做完了什么

- 把 14c 的两个能量按位置平均，给 118 个蛋白的 103,744 个位置贴上**结构关键 / 功能关键 / 两者 / 可以改**
- **对照 UniProt 注释**：功能关键富集 2.7 倍、两者都关键 9.3 倍，而结构关键被稀释（0.21 倍）——一个分数做不出这种分化
- **三个已成药的变构口袋**：KRAS switch-II 和 ABL1 肉豆蔻酸口袋都排进自己蛋白的前 23%（p ≈ 0.001）；SHP2 的构象开关型通道没信号，这是方法的边界
- 地图进了 S3 湖和 Glue（能用 Athena 查）、对外 API（`/v1/proteins/{protein}/sites`）和 agent（`get_site_map` 工具）

## 下一步

**14f · 药理伴侣**：这一章说了"折叠坏了可以试试扶住它的小分子"。14f 真的去验证一次——GLA 蛋白的突变里，哪些能被已上市的药 migalastat 救回来。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14e site map: what each position is for' && git log --oneline -1")